In [ ]:
import ast
from pathlib import Path
from collections import defaultdict
from typing import Set, Dict, List, Tuple

In [ ]:
def _collect_imports(tree: ast.AST) -> Dict[str, Tuple[int, str]]:
    """
    Walk the AST once and record every imported name.

    Returns
    -------
    dict mapping imported identifier → (line number where it was imported,
                                        original import statement text)
    """
    imports: Dict[str, Tuple[int, str]] = {}

    class ImportVisitor(ast.NodeVisitor):
        def visit_Import(self, node: ast.Import):
            # `import foo as bar`  →  name = bar (or foo if no alias)
            for alias in node.names:
                name = alias.asname or alias.name.split('.')[0]
                imports[name] = (node.lineno, f"import {alias.name}"
                                + (f" as {alias.asname}" if alias.asname else ""))

        def visit_ImportFrom(self, node: ast.ImportFrom):
            # `from pkg import a, b as c`
            module = ".".join(filter(None, [node.module]))
            for alias in node.names:
                # skip star‑imports – they are hard to analyse statically
                if alias.name == "*":
                    continue
                name = alias.asname or alias.name
                imports[name] = (
                    node.lineno,
                    f"from {module} import {alias.name}"
                    + (f" as {alias.asname}" if alias.asname else "")
                )

    ImportVisitor().visit(tree)
    return imports


def _collect_used_names(tree: ast.AST) -> Set[str]:
    """
    Walk the AST a second time and gather every identifier that is *read*.
    """
    used: Set[str] = set()

    class NameVisitor(ast.NodeVisitor):
        def visit_Name(self, node: ast.Name):
            # ctx can be Load (read), Store (assignment) or Del.
            # We only care about reads because an import that is only stored
            # (e.g. `import foo; foo = 5`) is effectively unused.
            if isinstance(node.ctx, ast.Load):
                used.add(node.id)
            self.generic_visit(node)

    NameVisitor().visit(tree)
    return used


def find_unused_imports(source_path: str) -> List[Tuple[int, str]]:
    """
    Main helper – returns a list of `(lineno, import_statement)` tuples
    for imports that never get used.
    """
    path = Path(source_path)
    if not path.is_file():
        raise FileNotFoundError(f"No such file: {source_path}")

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source, filename=str(path))

    imported = _collect_imports(tree)
    used = _collect_used_names(tree)

    unused = [
        (lineno, stmt)
        for name, (lineno, stmt) in imported.items()
        if name not in used
    ]

    # Sort by line number for nicer output
    unused.sort(key=lambda x: x[0])
    return unused

In [ ]:
# Path to the file you want to inspect – adjust as needed
target_file = "../utils/nostra_mapping.py"          # <-- replace with your filename

try:
    unused = find_unused_imports(target_file)
    if not unused:
        print(f"✅ No unused imports found in `{target_file}`.")
    else:
        print(f"⚠️ Unused imports detected in `{target_file}`:")
        for lineno, stmt in unused:
            print(f"  Line {lineno}: {stmt}")
except Exception as exc:
    print(f"❗ Error: {exc}")